In [40]:
import pandas as pd
import numpy as np
import json
pd.set_option('display.max_columns', 500)

MARKET = "eth_mapollo_usdc"
MARKET = "eth_rlp_usdc"


EVENTS_PATH = "/Users/yegortrussov/Documents/ml/lending_protocols/dataset_collection/data/markets_enriched"
HOURLY_MARKET_PATH = "/Users/yegortrussov/Documents/ml/lending_protocols/dataset_collection/data/markets_hourly_data"

df = pd.read_csv(f"/Users/yegortrussov/Documents/ml/lending_protocols/dataset_collection/data/markets_enriched/{MARKET}.csv")

df = pd.read_csv(f"{EVENTS_PATH}/{MARKET}.csv")
market_df = pd.read_csv(f"{HOURLY_MARKET_PATH}/{MARKET}.csv")

df.head(2)

/var/folders/hj/pbs977kd43s6n1l9z3mxrj200000gn/T/ipykernel_29309/1693465049.py:13: DtypeWarning:

Columns (6) have mixed types. Specify dtype option on import or set low_memory=False.

/var/folders/hj/pbs977kd43s6n1l9z3mxrj200000gn/T/ipykernel_29309/1693465049.py:15: DtypeWarning:

Columns (6) have mixed types. Specify dtype option on import or set low_memory=False.



,hash,type,timestamp,user_address,assets,assets_usd,liquidated_assets,liquidated_assets_usd,market,datetime,market_address,total_supply_before,total_borrow_before,total_supply_after,total_borrow_after,utilization_before,utilization_after,tx_actions,borrow_rate_before,supply_rate_before,borrow_rate_after,supply_rate_after,collateral_price,loan_asset_price,collateral_before,collateral_value_before,debt_before,supply_before,ltv_before,collateral_after,collateral_value_after,debt_after,supply_after,ltv_after,health_factor_before,health_factor_after,event_type,vault_flg,volatility_6h,drawdown_6h,trend_6h,volatility_24h,drawdown_24h,trend_24h,event_sequence_type
0,0x4ffb21a0b6ce460bf11305242993f70620690289339e...,MarketSupply,1761601475,0x000000000000000000000000000000000000dEaD,10729,0.010728,0,0.0,eth_rlp_usdc,2025-10-27 21:44:35,0xe1b65304edd8ceaea9b629df4c3c926a37d1216e2790...,4.765878e+07,4.300108e+07,4.765884e+07,4.300114e+07,0.902270,0.902270,1,0.066700,0.060215,0.066701,0.060215,1.253657,0.999898,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.010728,0.0,0.0,0.0,loan_position_supply,False,0.000214,-0.000459,0.000459,0.000129,-0.000599,0.000419,loan_position_supply
1,0x8d1e088aed77ed17840bc47e2735537a1d1d4f9debf2...,MarketSupply,1768843499,0x0000000f2eB9f69274678c76222B35eEc7588a65,10000000,9.997991,0,0.0,eth_rlp_usdc,2026-01-19 17:24:59,0xe1b65304edd8ceaea9b629df4c3c926a37d1216e2790...,8.325207e+07,7.591832e+07,8.325239e+07,7.591863e+07,0.911909,0.911909,1,0.101585,0.092671,0.101585,0.092671,1.279744,0.999799,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,9.997991,0.0,0.0,0.0,loan_position_supply,False,0.000888,-0.000525,-0.002052,0.000956,-0.000603,0.000015,loan_position_supply


In [41]:
yield_hist = pd.read_csv("/Users/yegortrussov/Documents/ml/lending_protocols/dataset_collection/data/tokens_yield_hist.csv")
yield_hist = yield_hist[yield_hist["symbol"] == "rlp"]

market_df['date'] = pd.to_datetime(market_df['datetime']).dt.date
yield_hist['date'] = pd.to_datetime(yield_hist['date']).dt.date
market_df = pd.merge(market_df, yield_hist[["date", "apy"]], on='date', how='left')


In [3]:
%load_ext autoreload

In [15]:
%autoreload 2
from utils import (
    create_hourly_user_dataset,
    plot_user_metrics,
)


In [16]:
df[df["vault_flg"] == False]["user_address"].value_counts()[:40]

user_address
0x0C36327e93F749a7eec04603410dF776150f47DE    6842
0x0F359FD18BDa75e9c49bC027E7da59a4b01BF32a    1254
0x000001ac4e512d670c34feDf6c71cE2F49fb160a     877
0xcB82e29930835A937429D4BCAD0a6fa6636755D7     581
0xE08D97e151473A848C3d9CA3f323Cb720472D015     500
0xe33Eaf6EE64f4B9353ff2ce3748FA05EEb9bd809     444
0xc6dD9976066F3364b4D6A72cD4F1fA0468327Aa7     325
0xba15E9b644685cB845aF18a738Abd40C6Bcd78eD     230
0x7193794ec82f527Efb618Ac50C078D348eCBA4b6     202
0xd6c757043e7d59088969B188923C62fa960aFE9B     197
0x1ed2584074B802929c7D94Ca43459AC9b074141d     175
0xBe7e92a1aa52Bf219979fB571c5d42a6B4107AbF     173
0x4a4037E53306f284cf214B733E9A54D6a5939818     172
0x9178eBE0691593184c1D785a864B62a326cc3509     168
0x55555815a5595991C3A0Ff119B59AEF6C8B55555     150
0x22F34f08406073bd27239cBaCF7942Ba0477704A     143
0x1597E4B7cF6D2877A1d690b6088668afDb045763     137
0x5dE1B7f14De50Ed6e430ea144eED8Bc9d0Bbb30C     124
0x478a1d816FB121fEf7263949d7869B896F2D569e     123
0x650bdBA05A0E6549

In [47]:
addr = "0xd7583E3CF08bbcaB66F1242195227bBf9F865Fda"

df[df["user_address"] == addr][[
    "datetime",
    "type",
    "debt_after",
    "collateral_value_after",
    "ltv_after",
    "event_type",
    "event_sequence_type",
    
    # "hash"
]].sort_values("datetime")[:50]

,datetime,type,debt_after,collateral_value_after,ltv_after,event_type,event_sequence_type
56730,2025-12-24 05:07:23,MarketSupply,0.000000e+00,0.000000e+00,0.000000,loan_position_supply,loan_position_supply
56731,2025-12-24 05:54:11,MarketSupply,0.000000e+00,0.000000e+00,0.000000,loan_position_supply,loan_position_supply
56732,2026-01-03 09:33:35,MarketSupply,0.000000e+00,0.000000e+00,0.000000,loan_position_supply,loan_position_supply
56733,2026-01-06 02:03:35,MarketWithdraw,0.000000e+00,0.000000e+00,0.000000,loan_position_withdraw,loan_position_withdraw
56734,2026-01-06 02:06:11,MarketSupplyCollateral,1.139639e+06,1.371325e+06,0.831313,position_open,position_open
56735,2026-01-06 02:06:11,MarketBorrow,1.139639e+06,1.371325e+06,0.831313,position_open,position_open
56736,2026-01-06 02:13:23,MarketSupplyCollateral,3.039037e+06,3.638458e+06,0.835519,borrow_more_w_collateral,borrow_more_w_collateral
56737,2026-01-06 02:13:23,MarketBorrow,3.039037e+06,3.638458e+06,0.835519,borrow_more_w_collateral,borrow_more_w_collateral
56738,2026-01-06 02:17:59,MarketSupplyCollateral,4.038721e+06,5.542247e+06,0.728946,borrow_more_w_collateral,borrow_more_w_collateral
56739,2026-01-06 02:17:59,MarketBorrow,4.038721e+06,5.542247e+06,0.728946,borrow_more_w_collateral,borrow_more_w_collateral


In [43]:
hourly_data = create_hourly_user_dataset(
    df,
    market_df,
    n_hours=0.5,
    threshold_date="2026-02-20",
)


here
(244, 45)


In [53]:
plot_user_metrics(hourly_data, ['apy', 'debt'], user_address=addr)

0.10861
17.383180000000003


,user_address,timestamp,datetime,collateral,debt,ltv,action,total_supply,total_borrow,market_utilization,borrow_rate,supply_rate,collateral_price,loan_asset_price,volatility_6h,drawdown_6h,apy
0,0x000001ac4e512d670c34feDf6c71cE2F49fb160a,1.742239e+09,2025-03-17 19:23:47,42679.000000,40793.268000,0.823837,position_open,1.691072e+07,1.145829e+07,0.677576,0.076526,0.051854,1.16,0.999827,0.003419,0.0,0.00000
1,0x000001ac4e512d670c34feDf6c71cE2F49fb160a,1.742241e+09,2025-03-17 19:53:47,42679.000000,40793.268000,0.823837,none,1.691072e+07,1.145829e+07,0.677576,0.076526,0.051854,1.16,0.999827,0.003419,0.0,0.00000
2,0x000001ac4e512d670c34feDf6c71cE2F49fb160a,1.742243e+09,2025-03-17 20:23:47,42679.000000,40793.268000,0.823843,none,1.691089e+07,1.149927e+07,0.679992,0.076497,0.052017,1.16,0.999835,0.003419,0.0,0.00000
3,0x000001ac4e512d670c34feDf6c71cE2F49fb160a,1.742245e+09,2025-03-17 20:53:47,42679.000000,40793.268000,0.823843,none,1.691089e+07,1.149927e+07,0.679992,0.076497,0.052017,1.16,0.999835,0.003419,0.0,0.00000
4,0x000001ac4e512d670c34feDf6c71cE2F49fb160a,1.742247e+09,2025-03-17 21:23:47,42679.000000,40793.268000,0.823843,none,1.691089e+07,1.149927e+07,0.679992,0.076497,0.052017,1.16,0.999835,0.003419,0.0,0.00000
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1477117,0xfe08d06A422992ab8eb3dBcD1475f39c8d595ca5,1.754200e+09,2025-08-03 05:39:59,48485.424579,43699.405318,0.738542,none,1.255933e+08,1.027994e+08,0.818510,0.086402,0.070722,1.22,0.999703,0.000000,0.0,4.08612
1477118,0xfe08d06A422992ab8eb3dBcD1475f39c8d595ca5,1.754201e+09,2025-08-03 06:09:59,48485.424579,43699.405318,0.738520,none,1.234344e+08,1.025944e+08,0.831166,0.087360,0.072611,1.22,0.999672,0.000000,0.0,4.08612
1477119,0xfe08d06A422992ab8eb3dBcD1475f39c8d595ca5,1.754203e+09,2025-08-03 06:39:59,48485.424579,43699.405318,0.738520,none,1.234344e+08,1.025944e+08,0.831166,0.087360,0.072611,1.22,0.999672,0.000000,0.0,4.08612
1477120,0xfe08d06A422992ab8eb3dBcD1475f39c8d595ca5,1.754205e+09,2025-08-03 07:09:59,48485.424579,43699.405318,0.738547,none,1.236041e+08,1.025953e+08,0.830031,0.087231,0.072405,1.22,0.999709,0.000000,0.0,4.08612
